In [14]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 40.73061,
	"longitude": -73.935242,
	"start_date": "2020-01-01",
	"end_date": "2025-12-31",
	"daily": ["temperature_2m_mean", "temperature_2m_max", "temperature_2m_min", "sunrise", "sunset"],
	"hourly": ["temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature", "precipitation", "rain", "snowfall", "snow_depth", "wind_speed_10m"],
	"timezone": "auto",
	"temperature_unit": "fahrenheit",
	"wind_speed_unit": "mph",
	"precipitation_unit": "inch",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_dew_point_2m = hourly.Variables(2).ValuesAsNumpy()
hourly_apparent_temperature = hourly.Variables(3).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(4).ValuesAsNumpy()
hourly_rain = hourly.Variables(5).ValuesAsNumpy()
hourly_snowfall = hourly.Variables(6).ValuesAsNumpy()
hourly_snow_depth = hourly.Variables(7).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(8).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["dew_point_2m"] = hourly_dew_point_2m
hourly_data["apparent_temperature"] = hourly_apparent_temperature
hourly_data["precipitation"] = hourly_precipitation
hourly_data["rain"] = hourly_rain
hourly_data["snowfall"] = hourly_snowfall
hourly_data["snow_depth"] = hourly_snow_depth
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m

hourly_dataframe = pd.DataFrame(data = hourly_data)

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_temperature_2m_mean = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_max = daily.Variables(1).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(2).ValuesAsNumpy()
daily_sunrise = daily.Variables(3).ValuesInt64AsNumpy()
daily_sunset = daily.Variables(4).ValuesInt64AsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["temperature_2m_min"] = daily_temperature_2m_min
daily_data["sunrise"] = daily_sunrise
daily_data["sunset"] = daily_sunset
daily_dataframe = pd.DataFrame(data = daily_data)


Coordinates: 40.738136291503906°N -73.91488647460938°E
Elevation: 14.0 m asl
Timezone: b'America/New_York'b'GMT-4'
Timezone difference to GMT+0: -14400s


In [15]:
traffic_data = pd.read_json("https://data.cityofnewyork.us/resource/7ym2-wayt.json")

In [16]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///data.db")

In [17]:
traffic_data.head(10)

,requestid,boro,yr,m,d,hh,mm,vol,segmentid,wktgeom,street,fromst,tost,direction
0,12512,Queens,2013,3,7,4,15,5,55135,POINT (1035363.4 185093.4),122 PL,SUTTER AV,ROCKAWAY BLVD,SB
1,12512,Queens,2013,3,7,4,30,8,55135,POINT (1035363.4 185093.4),122 PL,SUTTER AV,ROCKAWAY BLVD,SB
2,12512,Queens,2013,3,7,4,45,8,55135,POINT (1035363.4 185093.4),122 PL,SUTTER AV,ROCKAWAY BLVD,SB
3,12512,Queens,2013,3,7,5,0,7,55135,POINT (1035363.4 185093.4),122 PL,SUTTER AV,ROCKAWAY BLVD,SB
4,12512,Queens,2013,3,7,5,15,9,55135,POINT (1035363.4 185093.4),122 PL,SUTTER AV,ROCKAWAY BLVD,SB
5,12512,Queens,2013,3,7,5,30,4,55135,POINT (1035363.4 185093.4),122 PL,SUTTER AV,ROCKAWAY BLVD,SB
6,12512,Queens,2013,3,7,5,45,8,55135,POINT (1035363.4 185093.4),122 PL,SUTTER AV,ROCKAWAY BLVD,SB
7,12512,Queens,2013,3,7,6,0,13,55135,POINT (1035363.4 185093.4),122 PL,SUTTER AV,ROCKAWAY BLVD,SB
8,12512,Queens,2013,3,7,6,15,27,55135,POINT (1035363.4 185093.4),122 PL,SUTTER AV,ROCKAWAY BLVD,SB
9,12512,Queens,2013,3,7,6,30,17,55135,POINT (1035363.4 185093.4),122 PL,SUTTER AV,ROCKAWAY BLVD,SB


In [18]:
hourly_dataframe.head(10)

,date,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,snowfall,snow_depth,wind_speed_10m
0,2020-01-01 00:00:00+00:00,39.829998,79.699532,34.070000,32.075100,0.0,0.0,0.0,0.0,10.088845
1,2020-01-01 01:00:00+00:00,38.209999,75.076363,31.010000,29.407892,0.0,0.0,0.0,0.0,11.428421
2,2020-01-01 02:00:00+00:00,35.779999,77.071312,29.299999,26.927099,0.0,0.0,0.0,0.0,10.963583
3,2020-01-01 03:00:00+00:00,34.790001,78.990501,28.940001,26.067801,0.0,0.0,0.0,0.0,10.535296
4,2020-01-01 04:00:00+00:00,34.160000,77.779877,27.950001,24.650108,0.0,0.0,0.0,0.0,12.081872
5,2020-01-01 05:00:00+00:00,33.529999,77.149353,27.139999,24.095407,0.0,0.0,0.0,0.0,11.651742
6,2020-01-01 06:00:00+00:00,33.169998,74.576462,25.970001,23.780684,0.0,0.0,0.0,0.0,11.193944
7,2020-01-01 07:00:00+00:00,32.810001,70.202965,24.170000,23.302273,0.0,0.0,0.0,0.0,10.963583
8,2020-01-01 08:00:00+00:00,34.700001,75.006767,27.590000,26.143608,0.0,0.0,0.0,0.0,9.712292
9,2020-01-01 09:00:00+00:00,35.150002,70.469185,26.510000,26.498295,0.0,0.0,0.0,0.0,9.608688
